# Cora Link Prediction Dataset Preparation

Prepare Cora dataset for **link prediction** task using GWM-E architecture.

**Task:** Given two papers, predict if there's a citation link between them.

**Improvements:**
- ✅ MPNet embeddings (768D native, no padding)
- ✅ 2-hop neighborhoods (prevents oversmoothing)
- ✅ Domain-optimized for citation networks

---

## 1. Install Dependencies

PyTorch Geometric is not pre-installed on Kaggle.

In [ ]:
# Install PyTorch Geometric
!pip install -q torch-geometric torch-scatter torch-sparse

print("✓ Dependencies installed successfully")

## 2. Import Libraries

In [ ]:
import os
import sys
import json
import shutil
import torch
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from torch_geometric.utils import k_hop_subgraph
from huggingface_hub import hf_hub_download
from tqdm import tqdm

# Check environment
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Configuration

Adjust these parameters if needed.

In [ ]:
# Configuration
CONFIG = {
    'output_dir': '/kaggle/working/cora_gwm_data_link_prediction',
    'bert_model': 'sentence-transformers/all-mpnet-base-v2',  # UPGRADED: Better quality (768 dim)
    'num_hops': 2,  # REDUCED: Prevent oversmoothing (was 5)
    'sample_size': 5,  # Max neighbors to sample per hop
    'embedding_dim': 768,  # CHANGED: Native BERT dimension, no padding! (was 2048)
    'neg_sampling_ratio': 1.0,  # Number of negative samples per positive edge
    'val_size': 0.1,  # 10% validation split
    'test_size': 0.1,  # 10% test split (remaining 80% for training)
    'random_state': 42,  # For reproducibility
    'batch_size': 32,  # BERT encoding batch size
}

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG['device'] = device

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nTask: Link Prediction (Citation Network)")
print(f"Split: Train 80% / Val 10% / Test 10%")
print(f"\n📊 Using MPNet: High-quality general embeddings (768D)")
print(f"📈 Native embeddings: No padding!")
print(f"🔗 Negative sampling ratio: {CONFIG['neg_sampling_ratio']}")

## 4. Download Raw Cora Dataset

Download from HuggingFace Hub.

In [ ]:
print("="*70)
print(" "*20 + "STEP 1: Downloading Raw Data")
print("="*70)

repo_id = "Graph-COM/Text-Attributed-Graphs"
filename = "cora/processed_data.pt"
raw_dir = Path("/kaggle/temp/cora_raw")
raw_dir.mkdir(parents=True, exist_ok=True)
dest_file = raw_dir / "data.pt"

if dest_file.exists():
    print(f"✓ Raw data already exists at {dest_file}")
else:
    print(f"Downloading from {repo_id}...")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset"
    )
    
    print(f"Copying to {dest_file}...")
    shutil.copy(downloaded_path, dest_file)

# Load and verify
print("\nLoading dataset...")
data = torch.load(dest_file, weights_only=False)

print(f"\n✓ Successfully loaded Cora dataset!")
print(f"  Nodes: {data.num_nodes:,}")
print(f"  Edges: {data.edge_index.size(1):,}")
print(f"  Classes: {len(data.label_texts)}")
print(f"  Class names: {data.label_texts}")
print(f"  Has text: {hasattr(data, 'raw_texts')}")

if not hasattr(data, 'raw_texts'):
    raise ValueError("Dataset missing 'raw_texts' attribute!")

print(f"\nExample node text:")
print(f"  {data.raw_texts[0][:200]}...")

## 5. Generate BERT Embeddings

Encode all node texts using BERT.

In [ ]:
print("="*70)
print(" "*20 + "STEP 2: Generating BERT Embeddings")
print("="*70)

print(f"Loading BERT model: {CONFIG['bert_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['bert_model'])
bert_model = AutoModel.from_pretrained(CONFIG['bert_model']).to(device)
bert_model.eval()

print(f"✓ Model loaded on {device}")

# Generate embeddings
all_embeddings = []
batch_size = CONFIG['batch_size']

print(f"\nEncoding {len(data.raw_texts):,} texts (batch_size={batch_size})...")
with torch.no_grad():
    for i in tqdm(range(0, len(data.raw_texts), batch_size), desc="BERT encoding"):
        batch_texts = data.raw_texts[i:i+batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)
        
        # Get embeddings (mean pooling)
        outputs = bert_model(**encoded)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        all_embeddings.append(embeddings.cpu())

node_embeddings = torch.cat(all_embeddings, dim=0)
print(f"\n✓ Generated embeddings shape: {node_embeddings.shape}")

# NO PADDING OR TRUNCATION - use native dimension
current_dim = node_embeddings.shape[1]
target_dim = CONFIG['embedding_dim']

if current_dim != target_dim:
    print(f"\n⚠️  WARNING: Expected {target_dim}D but got {current_dim}D")
    print(f"   Adjusting to match actual model output...")
    CONFIG['embedding_dim'] = current_dim
else:
    print(f"\n✓ Using native embeddings: {current_dim} dimensions")
    print(f"   No padding! All dimensions contain real semantic information")

# Free GPU memory
del bert_model, tokenizer
if device == 'cuda':
    torch.cuda.empty_cache()
    print(f"✓ Freed GPU memory")

## 6. Create Multi-hop Graph Embeddings

Aggregate node embeddings from k-hop neighborhoods.

In [ ]:
print("="*70)
print(" "*20 + "STEP 3: Creating Multi-hop Embeddings")
print("="*70)

num_nodes = data.num_nodes
embedding_dim = node_embeddings.shape[1]
num_hops = CONFIG['num_hops']
sample_size = CONFIG['sample_size']

# Initialize multi-hop embeddings
multi_hop_embs = torch.zeros(num_hops, num_nodes, embedding_dim)

print(f"Building {num_hops}-hop neighborhoods for {num_nodes:,} nodes...")
print(f"Sampling up to {sample_size} neighbors per hop\n")

for node_idx in tqdm(range(num_nodes), desc="Creating multi-hop embeddings"):
    for hop in range(num_hops):
        if hop == 0:
            # Hop 0: Use the node's own embedding
            multi_hop_embs[hop, node_idx] = node_embeddings[node_idx]
        else:
            # Get k-hop subgraph
            subset, _, _, _ = k_hop_subgraph(
                node_idx=node_idx,
                num_hops=hop,
                edge_index=data.edge_index,
                relabel_nodes=False,
                num_nodes=num_nodes
            )
            
            # Exclude the center node itself
            neighbors = [n for n in subset.tolist() if n != node_idx]
            
            if len(neighbors) > 0:
                # Sample neighbors if there are too many
                if len(neighbors) > sample_size:
                    sampled_neighbors = torch.tensor(
                        np.random.choice(neighbors, sample_size, replace=False)
                    )
                else:
                    sampled_neighbors = torch.tensor(neighbors)
                
                # Aggregate neighbor embeddings (mean pooling)
                neighbor_embs = node_embeddings[sampled_neighbors]
                multi_hop_embs[hop, node_idx] = neighbor_embs.mean(dim=0)
            else:
                # No neighbors at this hop: use the node's own embedding
                multi_hop_embs[hop, node_idx] = node_embeddings[node_idx]

print(f"\n✓ Multi-hop embeddings shape: {multi_hop_embs.shape}")
print(f"  Expected: [{num_hops}, {num_nodes}, {embedding_dim}]")
print(f"\nEmbedding structure:")
print(f"  - Hop 0: Node itself")
print(f"  - Hop 1-{num_hops-1}: Aggregated k-hop neighbors")

## 7. Create Link Prediction Dataset

Split edges into train/val/test and generate negative samples for link prediction.

In [ ]:
print("="*70)
print(" "*20 + "STEP 4: Creating Link Prediction Data")
print("="*70)

# Extract edges
edge_index = data.edge_index
num_edges = edge_index.size(1)
print(f"Total edges in graph: {num_edges:,}")

# Create positive edge list (source, target pairs)
positive_edges = edge_index.t().numpy()  # Shape: [num_edges, 2]
print(f"Positive edges: {len(positive_edges):,}")

# Split edges into train/val/test
print("\nSplitting edges into train+val and test sets...")
train_val_edges, test_edges = train_test_split(
    positive_edges,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state']
)

print("Splitting train+val into train and val sets...")
val_size_adjusted = CONFIG['val_size'] / (1 - CONFIG['test_size'])
train_edges, val_edges = train_test_split(
    train_val_edges,
    test_size=val_size_adjusted,
    random_state=CONFIG['random_state']
)

print(f"\n✓ Edge split complete:")
print(f"  Training edges: {len(train_edges):,} ({len(train_edges)/num_edges*100:.1f}%)")
print(f"  Validation edges: {len(val_edges):,} ({len(val_edges)/num_edges*100:.1f}%)")
print(f"  Test edges: {len(test_edges):,} ({len(test_edges)/num_edges*100:.1f}%)")

# Generate negative samples (non-existent edges)
print("\n" + "="*70)
print("Generating negative samples...")
print("="*70)

def generate_negative_edges(positive_edges, num_nodes, num_negative, random_state=42):
    """Generate negative edges that don't exist in the graph."""
    np.random.seed(random_state)
    positive_set = set(map(tuple, positive_edges))
    negative_edges = []
    
    max_attempts = num_negative * 10  # Prevent infinite loop
    attempts = 0
    
    with tqdm(total=num_negative, desc="Sampling negative edges") as pbar:
        while len(negative_edges) < num_negative and attempts < max_attempts:
            # Random sample source and target
            src = np.random.randint(0, num_nodes)
            dst = np.random.randint(0, num_nodes)
            
            # Check: not self-loop and not existing edge
            if src != dst and (src, dst) not in positive_set:
                negative_edges.append([src, dst])
                pbar.update(1)
            
            attempts += 1
    
    if len(negative_edges) < num_negative:
        print(f"⚠️  Warning: Could only generate {len(negative_edges)} negative samples (requested {num_negative})")
    
    return np.array(negative_edges)

# Generate negative samples for each split
num_nodes = data.num_nodes
num_train_neg = int(len(train_edges) * CONFIG['neg_sampling_ratio'])
num_val_neg = int(len(val_edges) * CONFIG['neg_sampling_ratio'])
num_test_neg = int(len(test_edges) * CONFIG['neg_sampling_ratio'])

train_neg_edges = generate_negative_edges(positive_edges, num_nodes, num_train_neg, CONFIG['random_state'])
val_neg_edges = generate_negative_edges(positive_edges, num_nodes, num_val_neg, CONFIG['random_state'] + 1)
test_neg_edges = generate_negative_edges(positive_edges, num_nodes, num_test_neg, CONFIG['random_state'] + 2)

print(f"\n✓ Negative samples generated:")
print(f"  Train: {len(train_neg_edges):,}")
print(f"  Val: {len(val_neg_edges):,}")
print(f"  Test: {len(test_neg_edges):,}")

# Create conversations for link prediction
def create_link_conversations(pos_edges, neg_edges, data):
    """Create conversations for link prediction task."""
    conversations = []
    
    # Positive examples (label = "yes")
    for src, dst in tqdm(pos_edges, desc="Creating positive examples"):
        src_text = data.raw_texts[src]
        dst_text = data.raw_texts[dst]
        
        conversations.append({
            "id": [int(src), int(dst)],
            "conversations": [
                {
                    "from": "human",
                    "value": (
                        f"Does the following source paper cite the target paper? "
                        f"Answer with 'yes' or 'no' only.\n\n"
                        f"Source paper: {src_text}\n\n"
                        f"Target paper: {dst_text}"
                    )
                },
                {
                    "from": "gpt",
                    "value": "yes"
                }
            ],
            "graph": 1,
            "label": 1  # Positive link
        })
    
    # Negative examples (label = "no")
    for src, dst in tqdm(neg_edges, desc="Creating negative examples"):
        src_text = data.raw_texts[src]
        dst_text = data.raw_texts[dst]
        
        conversations.append({
            "id": [int(src), int(dst)],
            "conversations": [
                {
                    "from": "human",
                    "value": (
                        f"Does the following source paper cite the target paper? "
                        f"Answer with 'yes' or 'no' only.\n\n"
                        f"Source paper: {src_text}\n\n"
                        f"Target paper: {dst_text}"
                    )
                },
                {
                    "from": "gpt",
                    "value": "no"
                }
            ],
            "graph": 1,
            "label": 0  # Negative link
        })
    
    return conversations

print("\n" + "="*70)
print("Creating conversation data...")
print("="*70)

train_conversations = create_link_conversations(train_edges, train_neg_edges, data)
val_conversations = create_link_conversations(val_edges, val_neg_edges, data)
test_conversations = create_link_conversations(test_edges, test_neg_edges, data)

# Shuffle conversations
np.random.seed(CONFIG['random_state'])
np.random.shuffle(train_conversations)
np.random.shuffle(val_conversations)
np.random.shuffle(test_conversations)

print(f"\n✓ Created conversations:")
print(f"  Train: {len(train_conversations):,} ({len(train_edges):,} pos + {len(train_neg_edges):,} neg)")
print(f"  Val: {len(val_conversations):,} ({len(val_edges):,} pos + {len(val_neg_edges):,} neg)")
print(f"  Test: {len(test_conversations):,} ({len(test_edges):,} pos + {len(test_neg_edges):,} neg)")

print(f"\nClass balance:")
print(f"  Train: {len(train_edges)/(len(train_edges)+len(train_neg_edges))*100:.1f}% positive")
print(f"  Val: {len(val_edges)/(len(val_edges)+len(val_neg_edges))*100:.1f}% positive")
print(f"  Test: {len(test_edges)/(len(test_edges)+len(test_neg_edges))*100:.1f}% positive")

## 8. Create Multi-hop Embeddings for Edge Pairs

For each edge (src, dst), create combined multi-hop embeddings.

In [ ]:
print("="*70)
print(" "*20 + "STEP 5: Creating Edge Pair Embeddings")
print("="*70)

def create_edge_embeddings(conversations, multi_hop_embs):
    """
    Create embeddings for edge pairs by concatenating source and target multi-hop embeddings.
    Shape: [num_edges, 2 * num_hops, embedding_dim]
    """
    edge_embeddings = []
    
    for conv in tqdm(conversations, desc="Creating edge embeddings"):
        src_id, dst_id = conv['id']
        
        # Get multi-hop embeddings for source and target
        src_emb = multi_hop_embs[:, src_id, :]  # [num_hops, embedding_dim]
        dst_emb = multi_hop_embs[:, dst_id, :]  # [num_hops, embedding_dim]
        
        # Concatenate source and target embeddings
        edge_emb = torch.cat([src_emb, dst_emb], dim=0)  # [2*num_hops, embedding_dim]
        edge_embeddings.append(edge_emb)
    
    return torch.stack(edge_embeddings)  # [num_edges, 2*num_hops, embedding_dim]

print("Creating edge embeddings for all splits...")
train_edge_embs = create_edge_embeddings(train_conversations, multi_hop_embs)
val_edge_embs = create_edge_embeddings(val_conversations, multi_hop_embs)
test_edge_embs = create_edge_embeddings(test_conversations, multi_hop_embs)

print(f"\n✓ Edge embeddings created:")
print(f"  Train: {train_edge_embs.shape}")
print(f"  Val: {val_edge_embs.shape}")
print(f"  Test: {test_edge_embs.shape}")
print(f"\nEmbedding structure:")
print(f"  - First {CONFIG['num_hops']} hops: Source node multi-hop")
print(f"  - Last {CONFIG['num_hops']} hops: Target node multi-hop")
print(f"  - Total: {CONFIG['num_hops'] * 2} hops per edge")

print("\n" + "="*70)
print(" "*20 + "STEP 6: Saving Files")
print("="*70)

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}\n")

# Save train conversations
train_path = output_dir / "cora_train_link_data.jsonl"
with open(train_path, 'w', encoding='utf-8') as f:
    for conv in train_conversations:
        f.write(json.dumps(conv) + '\n')
train_size = os.path.getsize(train_path) / (1024**2)
print(f"✓ Saved: {train_path.name}")
print(f"  Samples: {len(train_conversations):,}")
print(f"  Size: {train_size:.2f} MB")

# Save validation conversations
val_path = output_dir / "cora_val_link_data.jsonl"
with open(val_path, 'w', encoding='utf-8') as f:
    for conv in val_conversations:
        f.write(json.dumps(conv) + '\n')
val_size = os.path.getsize(val_path) / (1024**2)
print(f"\n✓ Saved: {val_path.name}")
print(f"  Samples: {len(val_conversations):,}")
print(f"  Size: {val_size:.2f} MB")

# Save test conversations
test_path = output_dir / "cora_test_link_data.jsonl"
with open(test_path, 'w', encoding='utf-8') as f:
    for conv in test_conversations:
        f.write(json.dumps(conv) + '\n')
test_size = os.path.getsize(test_path) / (1024**2)
print(f"\n✓ Saved: {test_path.name}")
print(f"  Samples: {len(test_conversations):,}")
print(f"  Size: {test_size:.2f} MB")

# Save base node embeddings
node_emb_path = output_dir / "node_embeddings.pt"
torch.save(node_embeddings, node_emb_path)
node_emb_size = os.path.getsize(node_emb_path) / (1024**2)
print(f"\n✓ Saved: {node_emb_path.name}")
print(f"  Shape: {node_embeddings.shape}")
print(f"  Size: {node_emb_size:.2f} MB")

# Save multi-hop node embeddings (for reference)
multi_hop_path = output_dir / "multi_hop_node_embeddings.pt"
torch.save(multi_hop_embs, multi_hop_path)
multi_hop_size = os.path.getsize(multi_hop_path) / (1024**2)
print(f"\n✓ Saved: {multi_hop_path.name}")
print(f"  Shape: {multi_hop_embs.shape}")
print(f"  Size: {multi_hop_size:.2f} MB")

# Save edge embeddings for each split
train_edge_path = output_dir / "train_edge_embeddings.pt"
torch.save(train_edge_embs, train_edge_path)
train_edge_size = os.path.getsize(train_edge_path) / (1024**2)
print(f"\n✓ Saved: {train_edge_path.name}")
print(f"  Shape: {train_edge_embs.shape}")
print(f"  Size: {train_edge_size:.2f} MB")

val_edge_path = output_dir / "val_edge_embeddings.pt"
torch.save(val_edge_embs, val_edge_path)
val_edge_size = os.path.getsize(val_edge_path) / (1024**2)
print(f"\n✓ Saved: {val_edge_path.name}")
print(f"  Shape: {val_edge_embs.shape}")
print(f"  Size: {val_edge_size:.2f} MB")

test_edge_path = output_dir / "test_edge_embeddings.pt"
torch.save(test_edge_embs, test_edge_path)
test_edge_size = os.path.getsize(test_edge_path) / (1024**2)
print(f"\n✓ Saved: {test_edge_path.name}")
print(f"  Shape: {test_edge_embs.shape}")
print(f"  Size: {test_edge_size:.2f} MB")

# Calculate total size
total_size = (train_size + val_size + test_size + node_emb_size + 
              multi_hop_size + train_edge_size + val_edge_size + test_edge_size)

print(f"\n{'='*70}")
print(f"✅ ALL FILES SAVED SUCCESSFULLY!")
print(f"{'='*70}")
print(f"Total size: {total_size:.2f} MB")
print(f"Location: {output_dir}")

## 9. Summary and Download Instructions

In [ ]:
print("="*70)
print(" "*20 + "✅ PREPARATION COMPLETE!")
print("="*70)

print("\n📊 Summary:")
print(f"  • Dataset: Cora Citation Network")
print(f"  • Task: Link Prediction (binary classification)")
print(f"  • Total nodes: {data.num_nodes:,}")
print(f"  • Total edges: {num_edges:,}")
print(f"  • Train samples: {len(train_conversations):,} (80%)")
print(f"  • Val samples: {len(val_conversations):,} (10%)")
print(f"  • Test samples: {len(test_conversations):,} (10%)")
print(f"  • Edge embeddings: {train_edge_embs.shape[1:]} per edge")
print(f"  • Total output size: {total_size:.2f} MB")

print("\n📥 Download Instructions:")
print("  1. Click 'Output' tab on the right sidebar")
print("  2. Click 'Download All' to get cora_gwm_data_link_prediction.zip")
print("  3. Or download files individually:")

for file in sorted(output_dir.glob("*")):
    size_mb = os.path.getsize(file) / (1024**2)
    print(f"     • {file.name} ({size_mb:.2f} MB)")

print("\n🚀 Next Steps:")
print("  1. Download the files from Kaggle")
print("  2. Upload to your training environment")
print("  3. Update file paths in GWM-E training script")
print("  4. Run training with edge embeddings:")
print("     python train.py \\")
print("       --train_jsonl cora_train_link_data.jsonl \\")
print("       --val_jsonl cora_val_link_data.jsonl \\")
print("       --test_jsonl cora_test_link_data.jsonl \\")
print("       --train_edge_emb train_edge_embeddings.pt \\")
print("       --val_edge_emb val_edge_embeddings.pt \\")
print("       --test_edge_emb test_edge_embeddings.pt")

print("\n" + "="*70)

# Show example conversation
print("\n📝 Example Link Prediction Conversation:")
print("-"*70)
example = json.dumps(train_conversations[0], indent=2)
if len(example) > 800:
    example_dict = train_conversations[0]
    print(json.dumps({
        "id": example_dict["id"],
        "conversations": [
            {
                "from": example_dict["conversations"][0]["from"],
                "value": example_dict["conversations"][0]["value"][:300] + "..."
            },
            example_dict["conversations"][1]
        ],
        "graph": example_dict["graph"],
        "label": example_dict["label"]
    }, indent=2))
else:
    print(example)

print("\n" + "="*70)
print("Notebook completed successfully! 🎉")
print("="*70)

## Optional: Verify Files

In [ ]:
# Quick verification
print("Verifying saved files...\n")

# Load and check node embeddings
loaded_node_embs = torch.load(node_emb_path)
print(f"✓ Node embeddings: {loaded_node_embs.shape}")
assert loaded_node_embs.shape == node_embeddings.shape

# Load and check edge embeddings
loaded_train_edge = torch.load(train_edge_path)
print(f"✓ Train edge embeddings: {loaded_train_edge.shape}")
assert loaded_train_edge.shape == train_edge_embs.shape

loaded_val_edge = torch.load(val_edge_path)
print(f"✓ Val edge embeddings: {loaded_val_edge.shape}")
assert loaded_val_edge.shape == val_edge_embs.shape

loaded_test_edge = torch.load(test_edge_path)
print(f"✓ Test edge embeddings: {loaded_test_edge.shape}")
assert loaded_test_edge.shape == test_edge_embs.shape

# Load and check conversations
with open(train_path, 'r') as f:
    train_lines = f.readlines()
print(f"✓ Train conversations: {len(train_lines)} lines")
assert len(train_lines) == len(train_conversations)

with open(val_path, 'r') as f:
    val_lines = f.readlines()
print(f"✓ Val conversations: {len(val_lines)} lines")
assert len(val_lines) == len(val_conversations)

with open(test_path, 'r') as f:
    test_lines = f.readlines()
print(f"✓ Test conversations: {len(test_lines)} lines")
assert len(test_lines) == len(test_conversations)

print("\n✅ All files verified successfully!")
print(f"\nFinal split:")
print(f"  Train: {len(train_conversations):,} link examples")
print(f"  Val: {len(val_conversations):,} link examples")
print(f"  Test: {len(test_conversations):,} link examples")
print(f"  Total: {len(train_conversations) + len(val_conversations) + len(test_conversations):,}")